### pytorch를 이용하여 Doc2Vec 만들기

In [1]:
# !pip install torch

In [2]:
import re, collections
import torch
#nn은 모델들의 위치
import torch.nn as nn
#optim 옵티마이저
import torch.optim as optim
#데이터를 분할 (반복 학습하면서 학습 데이터, 검증 데이터 분할)
from torch.utils.data import Dataset, DataLoader

from konlpy.tag import Komoran

In [3]:
docs=[
    '나는 커피를 정말 좋아한다',
    '오늘 아침에 에스프레소 두 잔을 마셨다',
    '카페라떼가 제일 맛있다고 생각한다',
    '나는 차를 더 자주 마신다',
    '녹차를 마시면 기분이 편안해진다',
    '홍차는 향이 깊고 고급스러운 느낌이다',
    '카페에서 책을 읽는 시간이 너무 좋다',
    '허브티도 몸에 좋은 거 같다'
]

In [4]:
komoran=Komoran()

allow_pos=['NNP','NNG','VV','VA','MAG','SL']
stop_word=['하다','되다','이다','것','수','거']

def normalize(text):
    text=re.sub(r'[^가-핳0-9a-zA-Z]',' ', text)
    text=re.sub(r'\s+',' ',text)
    return text

def tokenize(text):
    text=normalize(text)

    tokens=[]
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            if pos in ['VV','VA']:
                word += '다'
            if word not in stop_word and len(word)>1:
                tokens.append(word)
    return tokens

In [5]:
tokenized_docs=[tokenize(doc) for doc in docs]
tokenized_docs

[['커피', '정말', '좋아하다'],
 ['오늘', '아침', '에스프레소', '마시다'],
 ['카페', '제일', '맛있다', '생각'],
 ['자주', '마시다'],
 ['녹차', '마시다', '기분', '편안', '진다'],
 ['차다', '깊다', '고급', '느낌'],
 ['카페', '읽다', '시간', '너무', '좋다'],
 ['좋다', '같다']]

In [6]:
#단어 사전 생성 -> Doc2Vec에서 최소 출현 횟수(min_count)의 영향
freq=collections.Counter(
    word for doc in tokenized_docs for word in doc
)

In [7]:
#단어 사전을 생성
#특수 토큰
    #<PAD> : 패딩의 의미 (단어의 개수가 달라서 단어별 벡터를 비교하는 경우에 개수를 맞춰주기 위한 특수 토큰)
    #<UNK> : 단어 사전에 없는 단어를 표기할 때 사용 (OOV)
#특수 토큰을 포함한 단어 사전을 생성 (min_count라는 제약)
min_count=1
vocab=['<PAD>','<UNK>'] + [word for word, cnt in freq.items() if cnt >= min_count]
vocab

['<PAD>',
 '<UNK>',
 '커피',
 '정말',
 '좋아하다',
 '오늘',
 '아침',
 '에스프레소',
 '마시다',
 '카페',
 '제일',
 '맛있다',
 '생각',
 '자주',
 '녹차',
 '기분',
 '편안',
 '진다',
 '차다',
 '깊다',
 '고급',
 '느낌',
 '읽다',
 '시간',
 '너무',
 '좋다',
 '같다']

In [8]:
#단어 사전의 단어와 위치 값을 이용햐여 dict 형태 데이터를 생성
stoi={
    word : idx for idx, word in enumerate(vocab)
}
stoi

{'<PAD>': 0,
 '<UNK>': 1,
 '커피': 2,
 '정말': 3,
 '좋아하다': 4,
 '오늘': 5,
 '아침': 6,
 '에스프레소': 7,
 '마시다': 8,
 '카페': 9,
 '제일': 10,
 '맛있다': 11,
 '생각': 12,
 '자주': 13,
 '녹차': 14,
 '기분': 15,
 '편안': 16,
 '진다': 17,
 '차다': 18,
 '깊다': 19,
 '고급': 20,
 '느낌': 21,
 '읽다': 22,
 '시간': 23,
 '너무': 24,
 '좋다': 25,
 '같다': 26}

In [9]:
#토큰화된 데이터르 stou의 데이터를 이용하여 인코딩
def encode(words):
    result=[stoi.get(word, stoi['<UNK>']) for word in words]
    return result

In [10]:
encodeed_docs=[encode(doc) for doc in tokenized_docs]
encodeed_docs

[[2, 3, 4],
 [5, 6, 7, 8],
 [9, 10, 11, 12],
 [13, 8],
 [14, 8, 15, 16, 17],
 [18, 19, 20, 21],
 [9, 22, 23, 24, 25],
 [25, 26]]

In [11]:
#학습 샘플 생성 (PV-DM, window 기반)
    #return (문서 ID, [주변 단어의 목록], 중심 단어)

def build_pvdm_samples(encode_data, window=2):
    #encode_data : 단어 사전을 통해 인코딩된 토큰 데이터
    samples=[]

    for d_id, doc in enumerate(encode_data):
        #d_id : 위치
        #doc : 인코딩된 데이터 원소
        if len(doc) < 2:
            continue
        for center in range(len(doc)):
            #center : 중심 단어의 위치
            left=max(0,center-window)
            right=min(len(doc), center+window+1)

            ctx=[]
            for i in range(left, right):
                if i !=center:
                    ctx.append(doc[i])
            if ctx:
                target=doc[center]
                samples.append(
                    (d_id, ctx, target)
                )
    return samples

samples=build_pvdm_samples(encodeed_docs)

In [12]:
#torch에 있는 linear 모델을 사용하기 위해 데이터의 형태로 알맞게 변환
#Dataset -> DataLoader에서 데이터를 가져가기 위한 class

class PVDMDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    # DataLoader 가 사용하는 특수 메서드 
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        d_id, ctx, target = self.samples[idx]
        return d_id, torch.tensor(ctx, dtype=torch.long), torch.tensor(target, dtype=torch.long)
    
def collate_fn(batch):
    #문맥에 따라서 길이가 다 다르기 때문에 리스트를 그대로 전달 
    doc_ids, ctx_lists, targets = zip(*batch)
    #batch -> [ [doc_id, ctx, target], [doc_id, ctx, target], ..... ]
    #zip(*batch) ->  [ [doc_id, doc_id,...], [ctx, ctx, ...], [target, target, ...] ]
    return torch.tensor(doc_ids, dtype=torch.long), list(ctx_lists), torch.stack(targets)

In [13]:

dataset = PVDMDataset(samples)

loader = DataLoader(
    dataset, batch_size=64, shuffle = True, collate_fn= collate_fn
)

In [14]:
import random

In [114]:
#모델을 정의
#딥러닝 모델은 생성자 함수 + 순전파(forward) 함수 선언

class Doc2VecPVDM(nn.Module):
    #생성자 함수 선언 -> 퍼셉트론 구조를 생성
    #단어별 임베딩
    #문장별 임베딩
    #선형 변환 층 생성
    #초기화 (자비에르)
    def __init__(
        self, vocab_size, doc_size, emb_dim=100
    ):
        super().__init__()

        #단어별 임베딩 -> vocab_size -> emb_dim
        self.word_emb = nn.Embedding(vocab_size, emb_dim)
        #문장별 임베딩 -> doc_size -> emb_dim
        self.doc_emb = nn.Embedding(doc_size, emb_dim)
        #단어의 확률 분포를 선형 변환층 생성하기 위해서 단어, 문서 임베딩을 이용
        self.proj = nn.Linear(emb_dim, vocab_size, bias=True)
        #자비에르 초기화 -> 임베딩 행렬의 초기값을 적당한 분포로 설정하여 학습이 잘 되도록 초기화하는 방법
        self._init_weight()
    #자비에를 초기화 함수 선언
    def _init_weight(self):
        nn.init.xavier_uniform_(self.word_emb.weight)
        nn.init.xavier_uniform_(self.doc_emb.weight)
        nn.init.xavier_uniform_(self.proj.weight)


    #순전파 함수 
    def forward(self, doc_ids, ctx_lists):
        #doc_idx : 문서들의 id 리스트
        #ctx_list : 주변 단어들의 목록
        '''
            batch ->
                {
                    'doc_idx' : tensor([0,1,2,3]),
                    'ctx_lists' : [tensor([3,4]), tensor([4,5])],
                    'targets' : tensor([2,3,4])
                }
        '''

        device=self.word_emb.weight.device

        #문서 임베딩 -> 특정 문서의 벡터 값
        #self.doc_emb() -> (문서의 개수, 임베딩 차원 수)
        #self.doc_emb(0) -> (임베딩 차원 수, )
        #self.doc_emb(doc_ids) -> (베치 사이즈, 임베딩 차원 수)
        dvec = self.doc_emb(doc_ids.to(device))

        #문서 임베딩의 평균 (dm_mean 매개변수가 1인 경우)
        ctx_mean=[]
        for ctx in ctx_lists:
            #ctx_lists : [tensor([3,4]), ...]
            #ctx : tensor([3,4])
            ctx = ctx.to(device)
            #단어들의 임베딩 대입
            we=self.word_emb(ctx)

            ctx_mean.append(
                #we : 2개의 행과 100개의 열로 이루어진 행렬 데이터
                #행의 값들의 평균
                we.mean(dim=0)
            )

        ctx_mean=torch.stack(ctx_mean, dim=0)

        #문서 임베딩과 같은 형태로 변경 -> (배치 사이즈, 임베딩 차원의 수)
        #Doc2Vec의 핵심 기능 : (문서 + 문맥) / 2 (단순 평균)
        mix =(dvec + ctx_mean) / 2.0

        #혼합이 된 mix를 Linear에 대입
        logits=self.proj(mix)

        return logits


    #Doc2Vec 핵심 메서드 : infer_vector() -> 새로운 문장을 임베딩
    def infer_vector(self, word_ids, epochs=100, lr=0.05):
        #새로운 문장에 대한 벡터 추론
        device = self.word_emb.weight.device

        #새로운 문장에 대한 정규 분포형 난수 벡터를 생성
        #학습에 대한 안정성, 수렴 속도 증가
        dvec = nn.Parameter(
            torch.randn(
                self.doc_emb.embedding_dim, device=device
            )
        )

        #하나의 임시 문장을 이용해서 반복학습을 하면서 단순 경사하강법을 이용하여 가중치의 변화를 준다.
        optimizer=optim.SGD([dvec], lr=lr)
        #손실 함수
        loss_fn=nn.CrossEntropyLoss()

        #간단 윈도우 기반 학습 샘플
        window=2
        triples=[]
        if (len(word_ids) <2):
            word_ids = word_ids * 2
        for center in range(len(word_ids)):
            left=max(0, center - window)
            right=min(len(word_ids), center + window + 1)
            ctx=[
                word_ids[i] for i in range(left,right) if i != center
            ]
            if ctx:
                triples.append(
                    (ctx, word_ids[center])
                )
        #문서 백터만 업데이트
        for _ in range(epochs):
            random.shuffle(triples)
            for ctx, target in triples:
                #문맥을 단어 임베딩의 평균화
                we = self.word_emb(torch.tensor(ctx, device=device))
                ctx_mean=we.mean(dim=0)
                #단순 평균
                mix=(dvec + ctx_mean) / 2.0
                #선형 모델에 저장
                logits=self.proj(mix)
                #로스 값 확인
                loss=loss_fn(logits.unsqueeze(0), torch.tensor([target], device=device))

                #기울기 초기화
                optimizer.zero_grad()
                #백워드 실행
                loss.backward()
                #스텝
                optimizer.step()    
        return dvec.detach().cpu()      #detach() : 해당 텐서를 연산 그래프엥서 분리
                                        #cpu() : 최종 결과를 cpu 메모리로 이동

In [115]:
emb=nn.Embedding(3,5)
print(emb.weight)

Parameter containing:
tensor([[ 0.5511,  0.6419,  0.8777,  0.9850,  2.4219],
        [-0.8389, -0.1887,  0.4662,  1.1136,  0.4686],
        [-1.4926, -0.9534,  0.1809,  0.6336, -0.0608]], requires_grad=True)


In [116]:
#자비에르 초기화
nn.init.xavier_uniform_(emb.weight)
print(emb.weight)

Parameter containing:
tensor([[-0.2455,  0.7770, -0.2802,  0.1007,  0.1691],
        [-0.3829,  0.5826, -0.2641, -0.1188, -0.1734],
        [ 0.3179,  0.1067,  0.0892,  0.7383, -0.8621]], requires_grad=True)


In [117]:
#cuda 사용이 가능한가?
device=torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
device

device(type='cpu')

In [118]:
model=Doc2VecPVDM(
    vocab_size=len(vocab),
    doc_size=len(docs),
    emb_dim=120
).to(device)

In [119]:
#옵티마이저 설정
optimizer=optim.Adam(model.parameters(), lr=3e-3)
#손실 함수 설정
loss_fn=nn.CrossEntropyLoss()

In [120]:
epochs = 50

for epoch in range(epochs):
    #모델을 학습 모드로 전환
    model.train()

    total_loss=0.0
    for doc_ids, ctx_lists, targets in loader:
        #doc_ids : 배치 사이즈만큼의 문서 id 목록
        #ctx_lists : 주변 단어들의 목록들 [tensor, tensor, ...]
        #targets : 중심 단어들의 목록 tensor([])

        #기울기 초기화
        optimizer.zero_grad()
        
        logits=model(doc_ids.to(device), ctx_lists)
        loss=loss_fn(logits, targets.to(device))
        loss.backward()
        optimizer.step()
        #로스 값을 누적
        total_loss += loss.item()

        #진행 상황을 로그 표시
        if (epoch + 1) % 10 == 0:
            print(f'{epoch + 1}회차 반복, total_loss는 {round(total_loss / len(loader), 4)}')

10회차 반복, total_loss는 2.966
20회차 반복, total_loss는 2.5596
30회차 반복, total_loss는 2.0961
40회차 반복, total_loss는 1.6361
50회차 반복, total_loss는 1.2797


In [121]:
@torch.no_grad()
def doc_vector(doc_id):
    result=model.doc_emb.weight[doc_id].detach().cpu()
    return result

#코사인 유사도 함수
def cos_sims(a, b):
    result = nn.functional.cosine_similarity(
        a.unsqueeze(0), b.unsqueeze(0), dim=1
    )
    return float(result)

In [122]:
doc_1, doc_2 = doc_vector(0), doc_vector(2)

print(
    round(
        cos_sims(doc_1, doc_2), 4
    )
)

-0.07


In [123]:
#infer_vector() : 새로운 문장을 벡터화하는 함수
new_sentence='오늘 카페에서 에스프레소를 마시며 책을 읽었다'
new_tokens=tokenize(new_sentence)
new_tokens

['오늘', '카페', '에스프레소', '마시다', '읽다']

In [124]:
new_ids=encode(new_tokens)
new_ids

[5, 9, 7, 8, 22]

In [125]:
new_vec=model.infer_vector(new_ids)

In [ ]:
new_vec.shape()

In [ ]:
#기존 학습에서 사용했던 문서들과의 유사도를 확인
for idx, text in enumerate(docs):
	sims=cos_sims(new_vec, doc_vector(idx))
	print(f'{idx} - {text} : 코사인 유사도 ({round(sims, 4)})')

0 - 나는 커피를 정말 좋아한다 : 코사인 유사도 (-0.0482)
1 - 오늘 아침에 에스프레소 두 잔을 마셨다 : 코사인 유사도 (0.2452)
2 - 카페라떼가 제일 맛있다고 생각한다 : 코사인 유사도 (-0.0033)
3 - 나는 차를 더 자주 마신다 : 코사인 유사도 (0.0469)
4 - 녹차를 마시면 기분이 편안해진다 : 코사인 유사도 (-0.0536)
5 - 홍차는 향이 깊고 고급스러운 느낌이다 : 코사인 유사도 (-0.1117)
6 - 카페에서 책을 읽는 시간이 너무 좋다 : 코사인 유사도 (0.1617)
7 - 허브티도 몸에 좋은 거 같다 : 코사인 유사도 (-0.0437)
